# Parte 1 – El modelo BG/NBD: verosimilitud y simulación

**Autor:** Herman Andrés Echeverría Rojas · Grupo 1

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

**Referencias principales**

- Fader, P. S., Hardie, B. G. S. y Lee, K. L. (2005). "Counting Your Customers" the Easy Way: An Alternative to the Pareto/NBD Model. *Marketing Science*, 24(2), 275–284.
- Fader, P. S., Hardie, B. G. S. y Lee, K. L. (2005). *Implementing the BG/NBD Model for Customer Base Analysis in Excel*. http://brucehardie.com/notes/004/

Este notebook contiene la derivación de la verosimilitud (1.1 y 1.2), la verificación del código de `src/bgnbd.py` y el estudio de simulación de recuperación de parámetros (1.3). El ajuste a CDNOW y su interpretación están en `02_ajuste_cdnow.ipynb`.

In [1]:
import sys, json, warnings
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datos import cargar_cdnow, resumen_clientes
from bgnbd import loglik_bgnbd, ajustar_bgnbd, PARAMS_ARTICULO, NOMBRES

trans = cargar_cdnow("../data/CDNOW_sample.txt")
res = resumen_clientes(trans, "1997-09-30", "1998-06-30")
x, tx, T = res["x"].values, res["tx"].values, res["T"].values
params = json.load(open("../resultados/params_cdnow.json"))
print(f"{len(res)} clientes; parámetros cargados de params_cdnow.json: {params}")

2357 clientes; parámetros cargados de params_cdnow.json: {'r': 0.243, 'alpha': 4.414, 'a': 0.793, 'b': 2.426}


## 1.1 Verosimilitud de un cliente

**Supuestos del BG/NBD** (Fader, Hardie y Lee, 2005, §3):

1. Mientras está vivo, el cliente compra según un proceso de Poisson con tasa $\lambda$; las esperas entre compras son exponenciales: $f(t_j \mid t_{j-1};\lambda) = \lambda e^{-\lambda (t_j - t_{j-1})}$.
2. $\lambda \sim \text{Gamma}(r, \alpha)$ entre clientes, con densidad $f(\lambda) = \dfrac{\alpha^r \lambda^{r-1} e^{-\alpha\lambda}}{\Gamma(r)}$ ($\alpha$ es una **tasa**).
3. Justo después de cada compra **repetida**, el cliente muere con probabilidad $p$.
4. $p \sim \text{Beta}(a, b)$ entre clientes, con densidad $f(p) = \dfrac{p^{a-1}(1-p)^{b-1}}{B(a,b)}$.
5. $\lambda$ y $p$ son independientes.

El tiempo se mide desde la primera compra del cliente ($t = 0$), así que solo se modelan las compras repetidas. Supongamos que el cliente hizo $x$ compras repetidas en $t_1 < t_2 < \dots < t_x$, todas dentro de $(0, T]$. Hay exactamente **dos historias** que producen esa observación:

**(a) Sigue vivo en $T$.** La primera compra en $t_1$ aporta $\lambda e^{-\lambda t_1}$. Cada compra siguiente exige haber sobrevivido a la anterior: $(1-p)\,\lambda e^{-\lambda(t_j - t_{j-1})}$. Después de la $x$-ésima compra también sobrevivió $(1-p)$ y no compró nada en $(t_x, T]$, lo que tiene probabilidad $e^{-\lambda(T - t_x)}$. Al multiplicar, los exponentes se suman de forma telescópica:

$$(1-p)^x\,\lambda^x\,e^{-\lambda T}.$$

**(b) Murió justo después de la compra en $t_x$.** Sobrevivió a las primeras $x-1$ compras, $(1-p)^{x-1}$, y murió en la $x$-ésima con probabilidad $p$. Después de morir ya no puede comprar, así que no hay factor de supervivencia en $(t_x, T]$:

$$p\,(1-p)^{x-1}\,\lambda^x\,e^{-\lambda t_x}.$$

Esta historia solo existe si $x > 0$: en el BG/NBD **no se puede morir antes de una compra repetida**. Por eso aparece $\delta_{x>0}$. Sumando las dos historias (son excluyentes):

$$L(\lambda, p \mid x, t_x, T) = (1-p)^x \lambda^x e^{-\lambda T} + \delta_{x>0}\; p(1-p)^{x-1} \lambda^x e^{-\lambda t_x}. \qquad (1)$$

Dos observaciones importantes:

- Los tiempos intermedios $t_1, \dots, t_{x-1}$ desaparecieron. La verosimilitud depende solo de **$(x, t_x, T)$**: frecuencia, recencia y antigüedad son estadísticos suficientes.
- Si $x = 0$, $L = e^{-\lambda T}$: la probabilidad de no comprar en $(0, T]$ estando vivo. **Este término no contiene a $p$**, así que los clientes sin compras repetidas no aportan información sobre la muerte. Esto vuelve a aparecer en 1.3.